# 1.3 CANN 架构与昇腾 950 NPU 原理
在通识课程 [cann_basics](../../../quick_start/cann_basics/README.md) 中，我们已经学习了 CANN 软件栈的基本结构与昇腾 NPU 的硬件基础（Host-Device 异构、AI Core 计算单元、存储层级等）。本节在此基础上，聚焦**昇腾 950 的第三代达芬奇（DaVinci）架构**——分离式 Cube/Vector 设计、双发射 Register-Based SIMD、同构 SIMT 混合编程、NDDMA 与 CV 融合等新特性。

已学习通识课程的同学可将本节视为架构升级的回顾与延伸；直接进入本课程的有基础开发者，本节也会自包含地介绍所需背景知识。本节依据《昇腾 950 NPU 架构白皮书》，深入拆解 950 芯片的体系结构、第三代 DaVinciCore 计算单元、存储层级与调度系统，为后续在 950 上编写 Ascend C 算子奠定硬件基础。


从 Ascend 950 这一代开始，向量处理单元在原有 SIMD（Single Instruction Multiple Data，单指令多数据）基础上引入了 SIMT（Single Instruction Multiple Threads，单指令多线程）能力。SIMD 是一条指令同时操作多个同构数据，适合矩阵乘、逐元素运算等规整高密度计算；SIMT 则是一条指令驱动多个独立线程，适合 Gather/Scatter 等离散访存和复杂分支场景。昇腾采用以 SIMD 为主、SIMT 为辅的同构编程模型，SIMD 支持全系列产品，SIMT 仅限 950。

| 模型 | 全称 | 核心思想 | 通俗比喻 | 适用场景 |
|---|---|---|---|---|
| SIMD | Single Instruction Multiple Data | 一条指令同时操作多个同构数据 | 一个厨师同时翻炒多份相同的菜 | 规整、高密度计算（矩阵乘、卷积、逐元素运算） |
| SIMT | Single Instruction Multiple Threads | 一条指令驱动多个独立线程 | 一个厨师指挥多个助手，各切各的菜 | 不规则访问、分支密集、稀疏计算（如 Gather/Scatter） |

> 关于 SIMD 与 SIMT 的详细对比和编程步骤，将在 [2.2 节](../02_operator_getting_started/02.02_simt_simd_introduction.ipynb) 中展开。

## 一、CANN 软件栈
CANN（Compute Architecture for Neural Networks）是华为针对 AI 场景推出的异构计算架构，对上支持多种 AI 框架（MindSpore、PyTorch、TensorFlow），对下服务 AI 处理器与编程，发挥承上启下的关键作用，是提升昇腾 AI 处理器计算效率的关键平台。

CANN 的核心设计理念是**分层抽象、软硬协同**：
- **框架适配层**：torch_npu 等适配插件，让 PyTorch/MindSpore 代码少量改动即可运行于 NPU。
- **算子库**：ops-math / ops-nn / ops-cv / ops-transformer 等预置高性能算子集合，覆盖从基础数学到 Transformer 的主流 AI 场景。
- **图引擎 GE**：整网图优化（算子融合、内存复用、自动流水）与计算图下沉执行，最大化端到端性能。
- **Ascend C 编程语言**：C/C++ 原生算子开发语言，开发者只需关注计算逻辑，由编译器和运行时自动完成指令调度、内存分配与并行切分。
- **毕昇编译器**：将 Ascend C 源码编译为 NPU 可执行二进制，自动完成指令融合、寄存器分配、流水线调度等优化。
- **运行时 Runtime（AscendCL）**：通过 AscendCL（Ascend Computing Language）C 语言接口提供 Device/内存/Stream/Event 管理与任务下发；在 950 上由 STARS2.0 调度系统实现全芯片任务调度。


<img src="./images/cann_software_architecture.png">


> 上图展示了 CANN 软件栈的分层结构：从上到下依次为框架适配层、算子库、图引擎 GE、Ascend C 编程语言、毕昇编译器、运行时 Runtime（AscendCL）和驱动，各层逐级抽象、软硬协同。


### 环境检查
若环境部署正常，可通过 `npu-smi info` 查看每张卡的功耗、温度、AICore 使用率、内存用量等信息。成功后会返回芯片版本字符串。


In [1]:
!npu-smi info


+------------------------------------------------------------------------------------------------+
| npu-smi 25.5.0                   Version: 25.5.0                                               |
+---------------------------+---------------+----------------------------------------------------+
| NPU   Name                | Health        | Power(W)    Temp(C)           Hugepages-Usage(page)|
| Chip                      | Bus-Id        | AICore(%)   Memory-Usage(MB)  HBM-Usage(MB)        |
+===========================+===============+====================================================+
| 6     910B3               | OK            | 87.3        39                0    / 0             |
| 0                         | 0000:0A:00.0  | 0           0    / 0          3438 / 65536         |
+===========================+===============+====================================================+
+---------------------------+---------------+----------------------------------------------------+
| NPU     

## 二、昇腾 950 芯片整体架构
950 采用多 Die 合封的 **Chiplet UMA（Unified Memory Access）** 架构，突破了单芯片面积与良率的物理限制：
- **2 个 AI Die**：每个 AI Die 内含多个 AI 子系统（AICore），是 AI 计算的核心载体。
- **2 个 IO Die**：负责片间互联与对外接口（HBM/PCIe/以太网）。
- **8 个（950PR）或 4 个（950DT）高速片上内存模块**：提供高带宽片上存储，配合 UMA 架构实现统一地址空间。

各 Die 之间通过 **D2D Clip（Die-to-Die 互联）** 实现高带宽低延迟通信。Chiplet UMA 架构的优势在于：
1. **可扩展性**：通过增加 Die 数量线性扩展算力与带宽。
2. **统一编程**：UMA 统一地址空间，开发者无需关心数据分布在哪个 Die 上。
3. **成本优化**：小面积 Die 良率更高，降低制造成本。


<img src="./images/NPU.png">


> 上图展示了昇腾 950 芯片的整体架构：采用多 Die 合封设计，包含 2 个 AI Die（计算核心）、2 个 IO Die（通信接口）和高速片上内存模块，各 Die 之间通过 D2D Clip 互联，构成 Chiplet UMA 统一内存架构。


### 2.1 Host 和 Device
在异构计算架构中，Host（CPU）负责运行时管理、任务下发与数据准备，Device（NPU）执行实际的 Kernel 计算任务。

#### 传统 Host-Device 协同的瓶颈
在传统异构计算中，即便 CANN 已支持将计算图整体下发到 Device 侧执行，Host 仍需深度参与**任务流的管理与调度**：为每条任务流分配资源、处理流间同步与依赖、响应 Device 侧的中断与状态上报等。当模型规模增大、任务流数量增多时，这些 Host 侧的调度开销占比越来越高，且 Host-Device 之间的控制消息往返延迟在微秒级，成为端到端时延的瓶颈。

#### 950 的 STARS2.0 任务流下沉
950 的 Host-Device 协同由 **STARS2.0**（Scheduling, Task Allocation and Runtime Synchronization）调度系统实现。STARS2.0 是全芯片的硬件任务和资源调度处理中心，核心改进是**支持 Host 将 2048 条任务流直接下沉到 Device 侧**：
- **一次性下发**：Host 将 2048 条任务流一次性写入 Device 侧的硬件任务队列，之后无需逐条参与调度。
- **Device 自主调度**：Device 侧的 STARS2.0 硬件调度器根据任务流配置自动预取任务、调度执行、上报结果给 Host，全程由软硬件协同完成，无需 Host 逐条介入。
- **ns 级调度延迟**：STARS2.0 通过专用高速控制总线（HSCB）与 AIC/AIV 交互，调度开销降低至 ns 级，相比传统 Host-Device 控制消息往返的微秒级延迟，大幅减少 Device 侧计算单元空闲间隙。
- **丰富调度能力**：STARS2.0 统一调度 AIC、AIV、AI CPU、DVPP、SDMA 等多个计算与搬运引擎，支持最多 128K 个同步标志位实现任务流间同步，支持 Group 调度与算力切分。

#### 效果示意
```
传统模式:  Host 下发任务流 → Device 执行 → Host 管理同步/依赖 → Device 执行 → ...
                        ↑ Host 持续参与任务流管理与同步 ↑

STARS2.0:  Host → 一次性下沉 2048 条任务流 → Device 硬件自主调度全部执行 → 上报结果给 Host
                        ↑ Host 只参与首尾，中间由 STARS2.0 硬件完成 ↑
```

对算子开发者而言，STARS2.0 意味着多核并行调度由系统自动完成，开发者只需关注单核 Kernel 逻辑与 Tiling 切分策略。


## 三、第三代 DaVinciCore（AI 子系统）
950 采用 **Cube/Vector 分离架构**（第三代 DaVinci），与前代（910B/910C 的 Cube+Vector 一体架构）有本质区别。每个 AI 子系统由 **1 个 AIC（Cube Core）** 和 **2 个 AIV（Vector Core）** 组成，共 36 个 AI 子系统。

分离架构的核心优势：
- **独立演进**：Cube 和 Vector 可分别优化算力/带宽，互不制约。
- **灵活配比**：不同算子对 Cube/Vector 需求不同，分离后可按需调度。
- **CV 融合**：打通 Cube L0C 与 Vector UB 的片上直连通路，矩阵乘结果无需经 Global Memory 中转即可送入 Vector 后处理。


<img src="./images/sram_arch.png">


> 上图展示了 950 单个 AI Core 子系统的内部微架构（对应白皮书图 4-1）。950 采用**一张量两向量**的分离式设计：中间列为 Cube 张量计算核心，左右两列为 Vector 向量计算核心（共 2 组，对称设计），顶部为 Bus Interface 总线接口。整颗 950 芯片共搭载 36 个这样的 AI Core 子系统。
>
> **Bus Interface（顶部）**：AI Core 与芯片内部高速总线、L2 统一缓存、HBM 主存的交互门户。同时它也是 **CV 融合通道**——Cube 矩阵乘结果可从 **L0C** 直接写入 Vector Core 的 UB，无需绕道 Global Memory。配套的 MTE（Memory Transfer Engine）数据搬运引擎负责在 GM 与片上 L1/UB 之间做异步搬运，搬运与计算完全并行。
>
> **中间列：Cube 张量计算子系统**
> - **Scalar0**：Cube 侧标量控制单元，负责指令调度、地址计算、循环控制，是 Cube 计算的指挥中心。
> - **L1 Buffer（512KB）**：Cube 一级缓存，暂存矩阵输入数据，作为主存与 L0 之间的中转，减少反复访问主存的开销。
> - **L0A Buffer（64KB）/ L0B Buffer（64KB）**：分别存放矩阵 A / B 数据，物理分离实现单周期同时读取两个矩阵输入，无端口冲突。
> - **Cube Core**：脉动阵列矩阵乘单元，单周期完成一次 16x16 x 16x16 的 FP16 矩阵乘加，等价于 **4096 次 FP16 乘加（MAC）**，是算力核心来源。
> - **L0C Buffer（256KB）**：矩阵乘输出/累加缓存，连续计算时可作累加输入，支持分块矩阵乘的中间结果暂存。
>
> **左右两列：Vector 向量计算子系统（共 2 组，对称设计）**
> - **Scalar1 / Scalar2**：各 Vector Core 内的标量控制单元，负责向量指令调度、地址生成、简单标量运算。
> - **Vector Core**：950 的核心升级——**双发射 Register-Based SIMD**。每个 Vector Core 内部集成**两条并行向量计算通路**，每条单周期可执行 64 次 FP32 或 128 次 FP16 乘加；双发射下单周期合计 **128 次 FP32 / 256 次 FP16 MAC**。前代仅为单通路 128 FP16/周期，950 实现向量算力翻倍。
> - **UB（Unified Buffer，256KB）**：向量/标量计算的专属片上缓存，向量计算的输入、输出、中间结果全部存放在 UB 中，是 Ascend C 算子开发中最核心的可编程存储。
> - **Register File（RegFile）**：紧邻 Vector ALU 的通用寄存器堆，在 UB 与 ALU 之间提供最低延迟的操作数存取。这是双发射 SIMD 的关键——操作数直接从 RegFile 读取（而非前代 VSIMD 的操作数队列），配合乱序执行（OoO）最大化指令吞吐。
>
> **Cube:Vector 算力配比 8:1 推导**：
> - 单个 Cube Core FP16 算力 = 16 x 16 x 16 = **4096 MAC/周期**
> - 单个 Vector Core FP16 算力 = 2 通路 x 128 = **256 MAC/周期**
> - 2 个 Vector Core 合计 = 256 x 2 = **512 MAC/周期**
> - Cube:Vector = 4096 : 512 = **8 : 1**
>
> 这一配比意味着矩阵乘算力远大于向量算力，因此 FlashAttention 等融合算子中向量部分（Scale、Mask、Softmax）容易成为瓶颈——950 通过双发射 Vector 和 CV 融合通道 有效缓解这一问题。


### 3.1 Cube Core（张量计算单元）
Cube Core 是脉动阵列矩阵乘单元，面向 GEMM、卷积、FlashAttention 等张量密集型任务，是 950 的算力核心来源。单周期完成一次 16x16 x 16x16 的 FP16 矩阵乘加，等价于 **4096 次 FP16 MAC/周期**。

**关键升级**：
- 新增 **HiF8/MXFP8/FP8/MXFP4** 低精度数据类型，MXFP4 峰值算力为上一代的 **4 倍**（同频下 HiF8/MXFP8/FP8 提供 2 倍 FP16 算力，MXFP4 提供 4 倍）。
- 更大 **L0C Buffer（256KB）**，提供更灵活的 Tiling 策略，减少中间结果回写。
- 支持 **L0C -> UB 随路量化（即在数据搬运过程中同时完成量化转换，无需额外指令）**：回写 UB 阶段直接完成 FP32 到 BF16/FP16/FP8 的量化与 NZ 到 ND/DN 的排布转换，降低核间带宽占用。
- 支持 **Cube->Vector 直通**（CV 融合），矩阵乘结果可通过 Bus Interface 直接送入 Vector Core 的 UB 做后处理，无需绕道 Global Memory。


<img src="./images/cube_core.png">


> 上图展示了 Cube Core 脉动阵列的原理，分上下两部分：
>
> **上半部分·数据流与时序**：矩阵 A 的行数据（x₀、x₁…）从左向右逐级传递，矩阵 B 的列数据（y₀、y₁…）从上向下逐级传递；每个交叉方块是一个 PE（处理单元），执行 `本单元累加值 = 左侧 x × 上方 y + 上方传入的累加值`，同时将 x 向右、新累加值向下传递——数据边流动边计算，最右侧 Σ 输出矩阵 C 的元素。图中为小规模阵列示意，真实 Cube Core 是 **16×16 的 PE 阵列（共 256 个 PE）**，稳态下单周期完成等价 **4096 次 FP16 乘加（MAC）**。
>
> **下半部分·PE 物理排布**：每个 PE 内部含乘法器、加法器与数据寄存器，整齐排成二维计算网格（图中 4×4 为简化画法，真实为 16 行×16 列）。这种二维网格完美匹配矩阵乘的二维数据特征，让计算与数据传输高度重叠，实现“数据不停、计算不停”的流水线效果。


<img src="./images/cubecore_data_type.png">


> 上图列出了 Cube Core 支持的数据类型矩阵：从传统的 FP32/FP16/BF16，到 950 新增的 **HiF8、MXFP8、FP8、MXFP4** 低精度格式。MXFP4 仅用 4 bit 表示一个元素，在同等硬件算力下可处理 2 倍于 FP8、8 倍于 FP16 的数据量，是大模型推理吞吐提升的关键。


### 3.2 Vector Core（向量计算单元）
950 的 Vector Core 采用**双模执行架构**——同一份 Vector Core 硬件上集成 **SIMD** 与 **SIMT** 两套执行流水线，这是第三代 DaVinci 的核心升级。对应白皮书「SIMD/SIMT 混合编程架构」，匹配 Ascend C 的 SIMD API 与 SIMT API 两套编程范式。

<img src="./images/vector_core_arch.png">

> 上图展示了 950 单个 Vector Core 的**双模执行架构全图**：左侧为 SIMD/SIMT 共享的公共通路（Scalar Unit 控制中枢、Async Function Queue 混合调度 SIMT/SIMD/NULL 函数、DMA 与 GM→Bus→UB 存储层级）；右侧上栏为 **SIMD 执行模式**（OoO 乱序发射 + 多 Bank UB + 访存合并 + 向量寄存器堆 + 双发射 Lane），右侧下栏为 **SIMT 执行模式**（Warp Scheduler + 顺序发射 + 线程独立寄存器 + 分支发散）。两种模式共享同一份 Vector Core 硬件，由编译器按算子特征选择执行模式。

#### 左侧·共享基础设施（SIMD/SIMT 共用）
- **Scalar Unit（标量单元）**：Vector Core 的控制中枢，负责地址计算、循环控制、分支判断与指令派发。
- **Async Function Queue（异步函数队列）**：队列中按序存放待执行函数，标注 SIMT/SIMD/NULL 三种类型，支持两种模式混合入队、异步调度，实现标量计算与向量计算的重叠并行。
- **DMA Unit + 存储层级**：DMA 独立于计算单元异步搬运，通路为 `Global Memory → Bus Interface → UB（Vector Cache/统一缓冲区）`，对应「搬运-计算-搬出」范式。950 将访存颗粒度从前代 512B 优化至 **128B**，对 SIMT 离散访存更友好。

#### 右上·SIMD 执行模式（经典向量模式，950 增强版）
取指调度：`I Cache → Program Sequence → OoO Dispatch（乱序发射）`。核心优化是 **OoO 乱序发射**——硬件自动分析指令依赖，无依赖指令打乱顺序并行发射，填满流水线槽位。
- **多 Bank UB**：UB 拆分为多个独立 Bank，支持多地址并行访问，减少 Bank 冲突。
- **Coalescing Unit（访存合并单元）**：将分散小块访存合并为连续大块访问，提升总线利用率。
- **Vector Register File**：紧邻计算单元的向量寄存器堆，延迟最低；配合**双发射**单周期可同时读取两组操作数。
- **向量执行单元**：由多个 Lane 组成，单指令控制所有 Lane 同步执行相同运算（单指令多数据）。单 Vector Core 单周期 **128 次 FP16 MAC**，双发射合计 **256 次 FP16 MAC/周期**；2 个 Vector Core 合计 **512 次 FP16 MAC/周期**，Cube:Vector = 4096:512 = **8:1**。适合逐元素运算、激活、规约等规整无分支算子。

#### 右下·SIMT 执行模式（950 新增，类 CUDA 线程级并行）
取指调度：`I Cache → Program Sequence → Warp Scheduler → In-order Dispatch（顺序发射）`。**Warp Scheduler** 将大量线程按固定大小组织成 Warp（线程束）调度执行，靠多 Warp 切换隐藏访存延迟（而非指令级乱序）。
- **SIMT Load/Store Unit**：适配线程级离散访存，配合访存合并单元将同一 Warp 内连续线程访存合并为大块访问。
- **SIMT Register File**：按线程独立分配寄存器上下文——每个线程都有自己的寄存器，支持线程独立分支与状态，这是 SIMT 与 SIMD 最本质的硬件区别。
- **分支发散（Branch Divergence）**：同一 Warp 内线程走不同分支时，硬件串行执行各分支路径再合并结果，保证复杂逻辑可执行性（传统 SIMD 无法做到）。适合复杂条件分支、不规则访存及从 CUDA 迁移的算子。


### 3.3 同构 SIMD/SIMT 混合编程
950 采用**以 SIMD 为主、SIMT 为辅**的混合编程模型：

- **SIMD（Single Instruction Multiple Data）**：承担大部分向量计算，借助双发 ALU 指令与乱序执行实现单指令多数据，高带宽高算力利用率。适合 Add、ReLU、Softmax 等逐元素或归约类算子。
- **SIMT（Single Instruction Multiple Thread）**：以线程为调度粒度，每个线程独立处理一个数据元素，适合 Gather、Scatter、条件分支等离散访存算子。

SIMD 和 SIMT 共享同一份 Vector Core 硬件，由编译器根据算子特征自动选择执行模式，开发者无需感知切换细节。这种同构设计避免了传统 GPU 中 SIMT 专用硬件的面积浪费。


### 3.4 CV 融合 与 NDDMA
- **CV 融合（Cube-Vector Fusion）**：打通 Cube **L0C**（矩阵乘输出累加缓存）与 Vector **UB** 之间的片上直连通路，矩阵乘结果从 L0C 直接写入 UB 做后处理，全程在片上流转、不必绕道 Global Memory。对 MatMul+激活、MatMul+LayerNorm、FlashAttention（MatMul+Softmax）等「矩阵乘+向量后处理」算子端到端提升显著。
  - **随路量化与排布转换**：L0C→UB 通路上固化硬件，搬运过程中自动完成精度转换（FP32→BF16/FP16/FP8/MXFP8/HiF8）与排布转换（Cube 输出的 NZ 格式→Vector 使用的 ND 格式），无需 Vector 核额外指令，零开销完成，进一步放大融合算子性能优势。
- **NDDMA（N 维直接内存访问）**：最多支持 5 维重排，GM→UB 一步完成搬运与排布转换，硬化地址生成。传统方案需要多次搬运+软件计算排布转换，NDDMA 将其压缩为单条指令，大幅减少带宽浪费与指令开销。


<img src="./images/cube_vector_fusion.png">


> 上图展示了 CV 融合的数据通路：前代路径为 Cube L0C → L1 → 总线 → Global Memory → 总线 → Vector UB，需一次 GM 往返（HBM 访问延迟约 100–300 cycle，往返达**数百 cycle**）；950 的 CV 融合路径为 **Cube L0C → Vector UB**（片上直连，完全不碰 GM），并在通路上随路完成精度转换（FP32→低精度）与排布转换（NZ→ND），零额外指令开销。MatMul+ReLU、FlashAttention 等「矩阵乘+向量后处理」融合算子可在片上一次完成，大幅减少中间数据回写。


- **NDDMA（N 维直接内存访问）**：最多支持 5 维重排，GM->UB 一步完成搬运与排布转换，硬化地址生成。传统方案需要多次搬运+软件计算排布转换，NDDMA 将其压缩为单条指令，大幅减少带宽浪费与指令开销。

<img src="./images/NDMA.png">


> 上图展示了 NDDMA 的工作原理：传统方案中，GM 到 UB 的搬运若涉及数据排布转换（如 ND->NZ），需要先搬运再做软件排布转换，多步完成；950 的 NDDMA 将搬运与最多 5 维的重排转换融合为**单条硬件指令**，硬化地址生成，一步到位。这在矩阵算子的数据准备阶段尤为关键——省去了大量中间搬运指令与带宽消耗。


## 四、存储层级
950 Memory 子系统采用多级存储，从近到远依次为：

| Memory | 位置 | 容量 | 说明 |
|---|---|---|---|
| L0A/L0B/L0C | Cube Core 内 | 小 | 矩阵乘操作数与累加缓冲 |
| UB（Unified Buffer） | Vector Core 内 | 中 | 向量计算操作数与结果缓冲 |
| L1 Buffer | AI 子系统内 | 大 | Cube/Vector 共享，CV 融合通道 |
| L2/L3 Cache | AI Die 内 | 更大 | 跨子系统共享缓存 |
| HBM / GM | 片上内存模块 | 最大 | 全芯片统一地址空间 |

**关键原则**：计算单元只能直接访问片上 Local Memory（L0/UB/L1），GM 中的数据需先经 DMA 搬入 Local Memory 方可参与计算。这一约束是 Tiling 切分的根本动因——将大 shape 数据分块搬入片上存储，计算完再搬回 GM。


<img src="./images/mem_arc.png">


> 上图展示了 950 的内存层次结构：从近到远依次为 AIC/AIV 内的 Local Memory（L0A/L0B/L0C/UB/L1）、L2 Cache（最高 128MB，服务 AI 计算）、L3 Cache（服务 AI CPU）、高速片上内存（950PR 最高 128GB / 950DT 最高 144GB）。计算单元只能直接访问 Local Memory，GM 数据需先经 DMA 搬入。


## 五、STARS2.0 全局硬件调度中枢
**STARS**（Scheduling, Task Allocation and Runtime Synchronization）是 950 新增的全局硬件级任务调度与同步中枢，所有调度逻辑固化为硬件，彻底替代前代依赖 CPU 软件调度的方案。Host 一次性下发任务后，后续调度、同步、流水衔接全由 STARS 硬件自动完成，调度延迟降至 ns 级。

**内部构成**：多通道 Task 任务队列（按引擎/优先级分流排队）+ Sched 中央调度器（按依赖与优先级分发、负载均衡）+ 四大功能组件——
- **Notify Sync**：硬件级同步原语，跨单元同步无需 CPU 介入，延迟从数百周期降至个位周期。
- **Conds**：基于“搬运完成/计算完成”等事件自动触发下一任务，实现“搬运→计算→搬出”全自动流水线。
- **Profiling**：原生硬件埋点自动采集执行数据，无需软件插桩，是性能调优核心数据源。
- **Fusion**：自动合并可连续执行的算子序列（如 MatMul+ReLU）为单个融合任务，消除多算子启动开销。

**下层总线**：**HSCB**（高速控制总线，低延迟）挂载 AIC/AIV 计算核心；**NoC**（片上网络，高带宽）挂载 UB DMA/SDMA/CCU/CPU/DVPP。

对算子开发者而言，无需手动管理多核调度与同步，只需用语言内置同步原语声明任务依赖，STARS 自动完成硬件级调度与流水衔接。


<img src="./images/starts2.png">


> 上图展示了 STARS2.0 调度系统架构：STARS 内部由多通道 Task 任务队列 + Sched 中央调度器 + 四大功能组件（Notify Sync 通知同步 / Conds 条件触发 / Profiling 硬件性能剖析 / Fusion 融合调度）构成，全部固化为硬件逻辑。下层通过两条独立总线对接硬件单元：**HSCB 高速控制总线**（低延迟）挂载 AIC/AIV 计算核心，**NoC 片上网络**（高带宽）挂载 UB DMA/SDMA/CCU/CPU/DVPP。Host 一次性下发任务后，STARS 硬件自主完成调度、同步与流水衔接，调度延迟降至 ns 级。


## 六、950 与 910B/910C 对比
| 特性 | 910B（A2） | 910C（A3） | 950 |
|---|---|---|---|
| 架构 | 第二代 DaVinci | 第二代 DaVinci | 第三代 DaVinci |
| Core 结构 | Cube+Vector 一体 | Cube+Vector 一体 | Cube/Vector 分离 |
| Vector 架构 | 传统 SIMD | 传统 SIMD | 双发射 Reg-Based SIMD |
| SIMT | 无 | 无 | 有（同构混合） |
| 低精度 | FP16/FP32 | FP16/FP32/BF16 | FP8/MXFP8/MXFP4/HiF8 |
| DMA | 传统 DMA | 传统 DMA | NDDMA（5 维重排） |
| 调度 | 传统调度 | 传统调度 | STARS2.0（2048 任务流） |
| 内存 | HBM | HBM | Chiplet UMA |

**对算子开发的影响**：950 的分离架构与双发射 SIMD 要求开发者重新思考 Tiling 策略与指令编排，但 Ascend C 编程接口保持一致，已有算子可平滑迁移。


---
## FAQ：STARS2.0 常见问题

**Q1：STARS 是什么？**
STARS 全称 Scheduling, Task Allocation and Runtime Synchronization，是昇腾 950 第三代达芬奇架构新增的全局硬件级任务调度与同步中枢，相当于整颗芯片的“任务指挥中心”。它统一调度片内所有计算、搬运、IO 引擎，实现硬件级的任务分发、同步、融合与性能监控，彻底替代前代依赖 CPU 软件调度的方案。架构与参数来自《昇腾 950 NPU 架构白皮书》及 CANN 9.1.0 官方文档，是 dav-3510 架构核心新增组件。

**Q2：STARS 相比前代调度的核心改进是什么？**
前代依赖 CPU 软件完成任务调度与同步，开销大、延迟高。STARS 将全部调度逻辑固化为硬件，主 CPU 只需一次性下发任务，后续调度、同步全由硬件自动完成，既释放 CPU 算力，又将调度延迟降低一个数量级（ns 级）。

**Q3：STARS 内部由哪些模块构成？**
由“多通道任务队列 + 中央调度器 + 四大功能组件”构成，全部固化为硬件逻辑：
- **多通道 Task 任务队列**：按硬件引擎/优先级分通道排队，支持多任务并行排队、多通道独立下发，避免单队列拥堵。
- **Sched 中央调度器**：按优先级与依赖关系取任务分发到硬件单元，管理任务强依赖，负责多 AI Core/多 DMA 引擎负载均衡。

**Q4：四大功能组件各自作用？**
- **Notify Sync（通知同步）**：硬件级同步原语，跨单元同步无需 CPU 介入或软件轮询，延迟从数百周期降至个位周期。典型场景：CV 融合中 Cube 完成后直接通知 Vector 启动。
- **Conds（条件触发）**：基于“搬运完成/计算完成/计数器达标”等事件自动触发下一任务，实现“搬运→计算→搬出”全自动流水线，开发者只需一次性配置依赖关系。
- **Profiling（硬件性能剖析）**：原生硬件埋点自动采集排队时间、执行时长、利用率等，无需软件插桩，是性能调优定位瓶颈的核心数据源。
- **Fusion（融合调度）**：自动识别可连续执行、无需中间回写主存的算子序列（如 MatMul+ReLU、MatMul+LayerNorm），合并为单个融合任务调度，消除多算子启动开销，配合 CV 融合最大化性能收益。

**Q5：HSCB 与 NoC 两条总线有何区别，各挂载什么单元？**
- **HSCB（高速控制总线）**：专为计算核心设计的低延迟控制总线，只传任务指令/同步信号不传大数据。挂载 **AIC**（AI 计算核心，含 Cube+Vector，全芯片 36 个，算力主体）与 **AIV**（专用向量加速引擎，分担向量/预处理负载）。
- **NoC（片上网络）**：通用型高带宽互联网络，负责大数据传输与外设调度，延迟略高于 HSCB。挂载 **UB DMA**（NDDMA 搬运引擎）、**SDMA**（系统 DMA，内存/主机间大块搬运）、**CCU**（缓存一致性单元）、**CPU**（控制核，驱动加载/任务下发，不参与核心计算）、**DVPP**（视频预处理引擎）。

**Q6：STARS 对算子开发者意味着什么？**
- 普通开发者无需手动管理多核调度与同步，只需用语言内置同步原语声明任务依赖，STARS 自动完成硬件级调度与同步。
- 做融合算子优化时可依赖 Fusion 调度能力，将连续算子合并为融合任务，减少任务启动与主存往返开销。
- 性能调优时可直接调用原生 Profiling 能力，获取硬件级性能数据，定位排队延迟、依赖等待等瓶颈。


---
## 课后练习
请根据本节课程学习内容完成以下题目进行自测，在每题下方的代码框中输入选项字母后运行。判断题 A=正确，B=错误。


**第1题**（判断题）950 的 Cube Core 和 Vector Core 是分离的独立硬件单元。
- A. 正确
- B. 错误


In [ ]:
q1 = ''  # 填入你的选项，如 'A'，修改后务必运行本单元格（Shift+Enter）
print(f'第1题答案已记录：{q1}' if q1 else '请填入答案并运行本单元格')


**第2题**（判断题）950 Vector Core 的双发射 SIMD 同一周期可执行两条独立指令。
- A. 正确
- B. 错误


In [ ]:
q2 = ''  # 填入你的选项，如 'A'，修改后务必运行本单元格（Shift+Enter）
print(f'第2题答案已记录：{q2}' if q2 else '请填入答案并运行本单元格')


**第3题**（单选题）950 的 NDDMA 最多支持几维重排？
- A. 3
- B. 4
- C. 5
- D. 6


In [ ]:
q3 = ''  # 填入你的选项，如 'A'，修改后务必运行本单元格（Shift+Enter）
print(f'第3题答案已记录：{q3}' if q3 else '请填入答案并运行本单元格')


**第4题**（单选题）950 的任务调度系统名称是？
- A. STARS
- B. STARS2.0
- C. DMA
- D. CANN


In [ ]:
q4 = ''  # 填入你的选项，如 'A'，修改后务必运行本单元格（Shift+Enter）
print(f'第4题答案已记录：{q4}' if q4 else '请填入答案并运行本单元格')


**全部作答完成后，运行下方代码查看批改结果：**


In [ ]:
import sys
sys.path.insert(0, './answer')
from grade import grade
grade(globals(), '01.03_answer.txt')
